# 01 | SOFC Data Audit and Time-Axis Reasoning

## Study objective

This notebook audits the raw EIS, IV and transient-response measurements collected from multiple SOFC cells under regular and randomized-redox operating regimes.

The analysis verifies data completeness, measurement consistency, cell identities, operating conditions and chronological ordering. It also establishes a reliable cell-specific time axis to prevent temporal leakage and create a traceable foundation for all subsequent preprocessing, feature engineering, degradation analysis and state-of-health forecasting.

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED = PROJECT_ROOT / "data" / "processed"

tables = {
    name: pd.read_parquet(PROCESSED / f"{name}.parquet") for name in ("eis", "iv", "transient")
}

manifest = json.loads((PROCESSED / "manifest.json").read_text(encoding="utf-8"))

audit = json.loads((PROCESSED / "audit_report.json").read_text(encoding="utf-8"))

print("Project root:", PROJECT_ROOT)
print("Processed directory:", PROCESSED)
print("Audit passed:", audit["passed"])
print("Source files:", manifest["source_file_count"])
print("Rows by signal:", manifest["rows_by_signal"])
print("Loaded tables:", list(tables.keys()))

## 1. Coverage before curves

In [ ]:
coverage = []

for signal, frame in tables.items():
    counts = frame[["cell_id", "assessment_index"]].drop_duplicates().groupby("cell_id").size()
    coverage.append(counts.rename(signal))

coverage = pd.concat(coverage, axis=1).fillna(0).astype(int)

coverage.loc["TOTAL assessments"] = coverage.sum()

coverage

In [ ]:
assessment_sets = {}

for signal, frame in tables.items():
    assessment_sets[signal] = {
        cell_id: set(group["assessment_index"]) for cell_id, group in frame.groupby("cell_id")
    }

missing_transient = []

for cell_id in sorted(assessment_sets["eis"]):
    eis_assessments = assessment_sets["eis"][cell_id]
    transient_assessments = assessment_sets["transient"][cell_id]

    missing = sorted(eis_assessments - transient_assessments)

    missing_transient.append(
        {
            "cell_id": cell_id,
            "missing_transient_assessments": missing,
        }
    )

pd.DataFrame(missing_transient)

In [ ]:
plot_data = (
    coverage.drop(index="TOTAL assessments")
    .rename_axis("cell_id")
    .reset_index()
    .melt(
        id_vars="cell_id",
        var_name="signal",
        value_name="assessments",
    )
)

fig, ax = plt.subplots(figsize=(11, 5))

sns.barplot(
    data=plot_data,
    x="cell_id",
    y="assessments",
    hue="signal",
    ax=ax,
)

ax.set(
    title="Repeated measurements available per cell",
    xlabel="Cell",
    ylabel="Assessment count",
)

ax.legend(title="Signal", ncols=3, frameon=False)

plt.tight_layout()

## 2. The two time axes

For observation $j$ in degradation assessment $k$ of cell $c$, write the signal as

$$y_{c,k,j}=f_c(k, t_{c,k,j})+\varepsilon_{c,k,j}.$$

Here $t_{c,k,j}$ is elapsed time **inside** one measurement, while $k$ orders repeated degradation assessments. Elapsed time commonly restarts near zero for each assessment. Consequently, concatenating all rows and pretending that `elapsed_s` is a global clock is mathematically wrong.

A row-random split is also optimistic: neighboring points from the same physical curve enter training and testing. The model can recognize a curve instead of generalizing to a future state or unseen cell.

In [ ]:
transient_bounds = (
    tables["transient"]
    .groupby(["cell_id", "assessment_index"])["elapsed_s"]
    .agg(start_s="min", end_s="max", samples="size")
    .reset_index()
)
transient_bounds.groupby("cell_id").agg(
    assessments=("assessment_index", "nunique"),
    median_start_s=("start_s", "median"),
    median_duration_s=("end_s", "median"),
    median_samples=("samples", "median"),
).round(3)

## 3. Inspect physical ranges and repairs

Cleaning rules are hypotheses about the measurement process. They should be narrow, testable, and traceable. This pipeline flags every source table that needed positional recovery instead of disguising the repair.

In [ ]:
ranges = pd.DataFrame(
    {
        "EIS frequency / Hz": tables["eis"]["frequency_hz"].describe(),
        "EIS real(Z) / ohm": tables["eis"]["z_real_ohm"].describe(),
        "IV voltage / V": tables["iv"]["voltage_v"].describe(),
        "IV current density / A cm^-2": tables["iv"]["current_density_a_cm2"].describe(),
        "Transient current density / A cm^-2": tables["transient"][
            "current_density_a_cm2"
        ].describe(),
    }
)
display(ranges.round(4))
display(pd.Series(manifest["schema_repaired_files"], name="repaired source file").to_frame())

In [ ]:
example = tables["transient"].query("cell_id == 'N1' and assessment_index in [1, 10, 20]")
fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(
    data=example,
    x="elapsed_s",
    y="current_density_a_cm2",
    hue="assessment_index",
    palette="viridis",
    ax=ax,
)
ax.set(
    title="Within-assessment dynamics change with degradation",
    xlabel="Elapsed time inside experiment / s",
    ylabel="Current density / A cm$^{-2}$",
)
plt.tight_layout()

## Conclusion

This notebook establishes a traceable and leakage-safe foundation for the SOFC degradation study. The EIS, IV and transient-response measurements must be treated as cell-specific, assessment-level observations rather than as one continuously sampled global time series.

Measurements are therefore ordered independently within each physical cell using the verified assessment sequence and elapsed-time information. Regular and randomized-redox cells are retained as separate operating regimes because their degradation histories are not directly interchangeable.

The three measurement modalities provide complementary information and should be aligned only when they belong to the same cell and assessment point. Missing modality records must be treated as genuine data availability limitations rather than being filled without physical justification.

All subsequent health indicators, temporal features and forecast targets must be calculated within each cell using only information available at the prediction time. Cell identity will be used for grouping and leakage-safe validation, while regime information will support stratified interpretation and performance reporting.

**Notebook decision:** The audited dataset and verified cell-specific time structure are suitable for physics-aware exploratory analysis. Any flagged records or unresolved alignment issues must remain excluded or explicitly documented in downstream notebooks.